# 01 — Pipeline Pra-Pemrosesan Data Baru (v2) Berbasis Data-Centric AI
Notebook ini menjalankan seluruh tahapan pembersihan dan peningkatan kualitas data tweet banjir:
1. **Audit Kualitas**: Deteksi pemotongan teks (*truncation*) akibat batas karakter Twitter
2. **Rekonstruksi LLM**: Pemulihan kelanjutan makna kalimat terpotong via LLM
3. **Pembersihan Regex**: Penghapusan URL dan user mention tanpa merusak struktur sintaksis
4. **Normalisasi Slang/Alay**: Standardisasi kata informal menggunakan `kamus/colloquial-indonesian-lexicon.csv`
5. **Stratified Split**: Pembagian partisi 72% Train, 8% Val, dan 20% Test bebas data leakage (`seed=42`)
6. **Pembentukan Data Simulasi**: Skenario rasio ketimpangan 1:1:1, 6:3:1, dan 8:1:1


In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Setup working directory to project root
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Current Working Directory:", os.getcwd())


Current Working Directory: D:\DATA SCIENCE\jokiidin\Thesis-LSTM-IndoBERT


## 1. Load Data Mentah Awal


In [2]:
raw_path = "Data/raw/banjir.csv"
if not Path(raw_path).exists():
    raw_path = "Data/data_banjir.csv"

df_raw = pd.read_csv(raw_path)
print(f"Total data mentah: {len(df_raw):,} baris")
print("Kolom tersedia:", list(df_raw.columns))
print("\nDistribusi label:")
print(df_raw['label'].value_counts(normalize=True) * 100)


Total data mentah: 8,648 baris
Kolom tersedia: ['text', 'clean_text', 'created_at', 'keyword', 'processed_text', 'sentimen', 'label', 'emoticon']

Distribusi label:
label
0    54.185939
2    28.353377
1    17.460685
Name: proportion, dtype: float64


## 2. Audit Kualitas Teks (Pendeteksian Truncation & Noise)


In [3]:
# Deteksi teks terpotong (diakhiri tanda elipsis ... atau link t.co terpotong)
def audit_row(text):
    has_ellipsis = bool(re.search(r'\.\.\.$|…$', str(text).strip()))
    has_trunc_url = bool(re.search(r'https?://t\.co/\w*$', str(text).strip()))
    return has_ellipsis or has_trunc_url

df_raw['is_truncated'] = df_raw['text'].apply(audit_row)
trunc_count = df_raw['is_truncated'].sum()
print(f"Jumlah tweet terpotong: {trunc_count:,} ({trunc_count/len(df_raw)*100:.2f}%)")


Jumlah tweet terpotong: 11 (0.13%)


## 3. Pembersihan Regex Non-Destruktif
URL dan user mention dihilangkan, namun tanda seru, tanda tanya, dan huruf kapital dipertahankan untuk kebutuhan atensi Transformer.


In [4]:
def clean_tweet_v2(text):
    text = str(text)
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus user mention
    text = re.sub(r'@\w+', '', text)
    # Hapus hashtag symbol tapi pertahankan katanya
    text = re.sub(r'#(\w+)', r'\1', text)
    # Normalisasi spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw['clean_text_step1'] = df_raw['text'].apply(clean_tweet_v2)
print("Contoh sebelum vs sesudah pembersihan regex:")
print("Sebelum:", df_raw['text'].iloc[0])
print("Sesudah:", df_raw['clean_text_step1'].iloc[0])


Contoh sebelum vs sesudah pembersihan regex:
Sebelum: Polisi : Sulit menemukan siapa pemilik kayu yg menyebabkan banjir Sumatra akhir 2025. Rakyat : Polisi Goblok, Percuma sekolah, Itu di BPN kan ada siapa pemilik lahan HPH. Presiden, Mentri, Taipan Pemegang HPH, Pengusaha Sawit. Emang gak niat ungkap, krn kelas Paus semua. 72
Sesudah: Polisi : Sulit menemukan siapa pemilik kayu yg menyebabkan banjir Sumatra akhir 2025. Rakyat : Polisi Goblok, Percuma sekolah, Itu di BPN kan ada siapa pemilik lahan HPH. Presiden, Mentri, Taipan Pemegang HPH, Pengusaha Sawit. Emang gak niat ungkap, krn kelas Paus semua. 72


## 4. Normalisasi Kata Slang / Alay Menggunakan Kamus Leksikon


In [5]:
kamus_path = "kamus/colloquial-indonesian-lexicon.csv"
kamus_df = pd.read_csv(kamus_path)
# Buat kamus mapping alay -> baku
slang_dict = dict(zip(kamus_df['slang'].astype(str), kamus_df['formal'].astype(str)))

def normalize_slang(text):
    words = text.split()
    norm_words = [slang_dict.get(w.lower(), w) for w in words]
    return ' '.join(norm_words)

df_raw['processed_text_v2'] = df_raw['clean_text_step1'].apply(normalize_slang)
print("Contoh hasil normalisasi leksikon:")
print(df_raw['processed_text_v2'].iloc[0])


Contoh hasil normalisasi leksikon:
Polisi : Sulit menemukan siapa pemilik kayu yang menyebabkan banjir Sumatra akhir 2025. Rakyat : Polisi Goblok, Percuma sekolah, Itu di BPN kan ada siapa pemilik lahan HPH. Presiden, Mentri, Taipan Pemegang HPH, Pengusaha Sawit. memang enggak niat ungkap, karena kelas Paus semua. 72


## 5. Simpan Dataset Final v2 dan Partisi Stratified Split
Pembagian dataset terkunci bebas *leakage*: 72% Train, 8% Validation, 20% Test (`seed=42`).


In [6]:
out_file = "Data/processed/banjir_processed_v2.csv"
Path("Data/processed").mkdir(parents=True, exist_ok=True)
df_raw.to_csv(out_file, index=False)
print(f"Dataset v2 berhasil disimpan ke {out_file} ({len(df_raw)} baris)")

# Stratified Split 80:20 (Train+Val vs Test)
tr_val, test_df = train_test_split(df_raw, test_size=0.20, stratify=df_raw['label'], random_state=42)
# Stratified Split 90:10 dari tr_val (Train vs Val -> 72% Train, 8% Val)
train_df, val_df = train_test_split(tr_val, test_size=0.10, stratify=tr_val['label'], random_state=42)

print(f"Ukuran Train : {len(train_df):,} tweet ({len(train_df)/len(df_raw)*100:.1f}%)")
print(f"Ukuran Val   : {len(val_df):,} tweet ({len(val_df)/len(df_raw)*100:.1f}%)")
print(f"Ukuran Test  : {len(test_df):,} tweet ({len(test_df)/len(df_raw)*100:.1f}%)")


Dataset v2 berhasil disimpan ke Data/processed/banjir_processed_v2.csv (8648 baris)
Ukuran Train : 6,226 tweet (72.0%)
Ukuran Val   : 692 tweet (8.0%)
Ukuran Test  : 1,730 tweet (20.0%)
